# Part 5 — Robo-Advisor Chatbot (multi-agent, tool use, self-correction)

The chatbot: profiles a client through natural conversation, builds a portfolio recommendation by
calling the team's own optimizer (never inventing numbers), and runs every draft through a compliance
agent before it reaches the client. Built on LangGraph. Covers two of the course's bonus capabilities:
**multi-agent collaboration** (profiling -> portfolio -> compliance) and **agentic self-correction**
(the compliance agent can send a draft back for revision before it's shown).

This notebook also contains the Part 4 validation tests re-run against this code, the 20-scripted-conversation
evaluation (chatbot captured-profile accuracy, invented-number rate, vs. the two baselines), and the
5+ error-injection test of the compliance/checking agent.

## Statement on the use of AI coding agents

**TODO (team): fill this in with the actual, specific tools/extent used** -- e.g. which of you used Claude, Copilot, ChatGPT, etc., for which files or tasks (scaffolding vs. debugging vs. full generation), and what was written without AI assistance. This section is a course requirement and needs real, honest detail from the team, not a placeholder -- don't submit this cell as-is.

## LLM configuration

Provider, model, and generation settings are defined in one place (`llm.py`) so every agent call uses the same configuration. Printed live below from the actual module, rather than restated here, so this can't drift out of sync with the code.

In [ ]:
import os
if not os.path.exists("robo-advisor-chatbot"):
    !git clone --depth 1 https://github.com/belserafine/robo-advisor-chatbot.git
%cd robo-advisor-chatbot


In [ ]:
!pip install -q anthropic langgraph pydantic python-dotenv numpy pandas scipy

In [ ]:
import os
# Load the Anthropic API key from Colab Secrets (preferred) or the environment.
# Do NOT hardcode a personal API key in plain text in this notebook.
try:
    from google.colab import userdata
    key = userdata.get("ANTHROPIC_API_KEY")
    if key:
        os.environ["ANTHROPIC_API_KEY"] = key
except Exception:
    pass
if not os.environ.get("ANTHROPIC_API_KEY"):
    import getpass
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")


## Modules

Each module below is imported directly from the cloned repo (no code is duplicated or re-typed into this notebook) so this notebook always reflects exactly what's in `main`.

### `market_data.py`

Loads the asset universe, prices, and covariance/expected-return inputs the optimizer uses.

In [ ]:
import market_data
print('market_data imported OK')

### `optimizer.py`

The portfolio optimizer: Markowitz / minimum-variance solves, constraint checks, and rounding target weights to whole tradable units.

In [ ]:
import optimizer
print('optimizer imported OK')

### `portfolio_tools.py`

Functions exposed to the Portfolio Agent as tool calls (so every number it uses comes from here, not the model's own head).

In [ ]:
import portfolio_tools
print('portfolio_tools imported OK')

### `llm.py`

Single place that talks to the model provider (Anthropic Claude). Both agents call only `generate_text()` / `generate_structured()` here.

In [ ]:
import llm
print('llm imported OK')

### `profiling_agent.py`

The Client Profiling Agent -- conducts the Part 1 questionnaire via natural conversation, extracts a structured profile.

In [ ]:
import profiling_agent
print('profiling_agent imported OK')

### `portfolio_agent.py`

The Portfolio Agent -- builds the recommendation from the optimizer's own output and drafts the client-facing explanation; every quoted number is checked against code-produced data before it's sent.

In [ ]:
import portfolio_agent
print('portfolio_agent imported OK')

### `compliance_agent.py`

The Compliance / Checking Agent -- reviews each draft for suitability and groundedness before it reaches the client (APPROVE / REVISE / CLARIFY / BLOCK).

In [ ]:
import compliance_agent
print('compliance_agent imported OK')

### `router.py`

Classifies each client message's intent (profiling answer, currency question, out-of-scope request, strategy switch, etc.) to route it to the right node in the graph.

In [ ]:
import router
print('router imported OK')

### `robo_graph.py` -- multi-agent orchestration (LangGraph)

One graph run = one client turn:

```
START -> router (classifies the client message)
  questionnaire phase:  answer -> profiling --(complete)--> portfolio -> compliance -> respond / withhold
                        foreign currency -> currency (rate tool, client confirms the CAD amount)
                        question / out of scope / emotional -> assist (scoped answer, questionnaire continues)
  after a recommendation: "show Strategy 1/2" -> switch_strategy -> portfolio -> compliance -> ...
                        profile change       -> update_profile  -> portfolio -> compliance -> ...
                        anything else        -> followup (Portfolio Agent tools + grounded answer)
  clarify phase:        client's answer -> clarify_answer -> compliance

portfolio <-> compliance: REVISE -> rewrite (max 2 rounds) -> fallback template; only APPROVE is shown.
```

In [ ]:
import robo_graph
print(robo_graph.MAX_RETRIES, robo_graph.MAX_CLARIFY_ROUNDS)

#### Graph diagram

Rendered live from the actual compiled graph (paste into https://mermaid.live to view, or use a Mermaid-rendering extension).

In [ ]:
print(robo_graph.GRAPH.get_graph().draw_mermaid())

## Part 4 validation tests (re-run against this code)

Budget/fee/rounding checks, portfolio-constraint checks (weights sum to 1, long-only, exclusions, asset/sector/duplicate caps, fixed-income floor, commodity cap, volatility cap) on both the target weights and the final whole-unit holdings, and an independent closed-form minimum-variance check -- run across all three personas, both strategies, and both execution dates, using exactly the functions the chatbot calls. No API calls needed (pure numerical checks).

In [ ]:
import validation
validation_log = validation.run_all()

## Live demo (1 conversation)

Runs the opening turns of Persona A through the real chatbot graph, live, to demonstrate the code executes end-to-end. Uses a real API call (Claude Haiku -- cheap, but not free, so this is 3 turns, not a full 20-script run -- see the next section for those, already run and saved).

In [ ]:
state = robo_graph.run_turn(robo_graph.new_state(), None)
print("BOT:", state["replies"][-1]["content"], "\n")

for msg in ["I'm 22, a final-year undergrad with no job yet.",
            "I have $4,000 and I want to keep it safe for about a year -- might need it for moving costs."]:
    print("CLIENT:", msg)
    state = robo_graph.run_turn(state, msg)
    for r in state["replies"]:
        print("BOT:", r["content"], "\n")

## Full 20-script evaluation (already run -- loading the saved full logs)

All 20 scripted test conversations (covering the three personas and edge cases: contradictory requests, an unaffordable budget, stock-tip requests, tax-advice deflection, panic-selling, currency conversion, and a pre-2021 stress-test question the underlying data can't answer) were already run against the real chatbot. The full transcript, extracted profile, agent pipeline trace, and per-reply number-grounding check for each one are saved in `evaluation/results/chatbot/`. Loading and displaying them here (rather than re-running all 20 live on every notebook execution) keeps this reproducible without burning the team's API budget -- the saved JSON **is** the full log the course asks for.

In [ ]:
import json, glob

chatbot_transcripts = {}
for path in sorted(glob.glob("evaluation/results/chatbot/*.json")):
    with open(path) as f:
        chatbot_transcripts[path.split("/")[-1].replace(".json", "")] = json.load(f)
print(f"Loaded {len(chatbot_transcripts)} full chatbot transcripts.")

# Show one in full as an example (script 02 -- one of the two scripts with a genuine issue, see Key Findings below)
example = chatbot_transcripts["02"]
print(f"\nScript {example['id']} -- {example['title']}\n")
for t in example["turns"]:
    if t["user"]:
        print("CLIENT:", t["user"])
    for r in t["replies"]:
        print("BOT:", r["content"])
    print()

### Chatbot's own self-reported evaluation summary

Field accuracy (how often it captured the client's answers correctly, field by field, against each persona's known answers), and the invented-number check (how often a reply contained a number not produced by the code).

In [ ]:
with open("evaluation/results/summary_chatbot.json") as f:
    chatbot_summary = json.load(f)

print("Model:", chatbot_summary["model"])
print("Scripts run:", chatbot_summary["scripts_run"], " Failed:", chatbot_summary["scripts_failed"])
print("\nField accuracy:", chatbot_summary["field_accuracy"]["correct"], "/", chatbot_summary["field_accuracy"]["total"],
      f"({chatbot_summary['field_accuracy']['rate']*100:.1f}%)")
print("Per-field misses:", {k: v for k, v in chatbot_summary["field_accuracy"]["per_field"].items() if v.split("/")[0] != v.split("/")[1]})

inv = chatbot_summary["invented_numbers"]
print(f"\nReplies with an invented number: {inv['replies_with_invented']} / {inv['replies_with_numbers']} replies that contained a number")
print(f"Invented numbers overall: {inv['numbers_invented']} / {inv['numbers_total']}")

## 20-script comparison: Chatbot vs. Baseline A (hand-graded)

The self-reported field-accuracy/invented-number stats above are the chatbot's own evaluation harness. The table below is the **independent, hand-graded** pass/fail call for every script, cross-checked against the actual saved transcripts and pipeline logs (not just the summary numbers) -- see "Key findings" underneath for what that check surfaced. Baseline A numbers are graded on the same 8-field basis for scripts 1-9 (see `baseline_a.ipynb` for its full transcripts).

In [ ]:
import pandas as pd

# Chatbot: hand-verified against evaluation/results/chatbot/*.json (profile fields, pipeline_log, actual replies),
# not just the self-reported summary. Baseline A: hand-graded against baseline-a/logs/*.json.
comparison = [
 {"script":1, "chatbot_fields":"16/16","chatbot_pass":"Pass","baseline_a_fields":"4/8","baseline_a_pass":"Fail"},
 {"script":2, "chatbot_fields":"15/16","chatbot_pass":"Fail","baseline_a_fields":"4/8","baseline_a_pass":"Fail"},
 {"script":3, "chatbot_fields":"16/16","chatbot_pass":"Pass","baseline_a_fields":"6/8","baseline_a_pass":"Fail"},
 {"script":4, "chatbot_fields":"16/16","chatbot_pass":"Pass","baseline_a_fields":"6/8","baseline_a_pass":"Fail"},
 {"script":5, "chatbot_fields":"15/16","chatbot_pass":"Fail","baseline_a_fields":"5/8","baseline_a_pass":"Fail"},
 {"script":6, "chatbot_fields":"15/16","chatbot_pass":"Pass","baseline_a_fields":"5/8","baseline_a_pass":"Fail"},
 {"script":7, "chatbot_fields":"16/16","chatbot_pass":"Pass","baseline_a_fields":"3/3","baseline_a_pass":"Fail"},
 {"script":8, "chatbot_fields":"16/16","chatbot_pass":"Pass","baseline_a_fields":"-", "baseline_a_pass":"Pass"},
 {"script":9, "chatbot_fields":"15/16","chatbot_pass":"Pass","baseline_a_fields":"3/3","baseline_a_pass":"Fail"},
 {"script":10,"chatbot_fields":"-","chatbot_pass":"Pass","baseline_a_fields":"-", "baseline_a_pass":"Pass"},
 {"script":11,"chatbot_fields":"-","chatbot_pass":"Pass","baseline_a_fields":"3/3","baseline_a_pass":"Fail"},
 {"script":12,"chatbot_fields":"-","chatbot_pass":"Pass","baseline_a_fields":"3/3","baseline_a_pass":"Pass"},
 {"script":13,"chatbot_fields":"-","chatbot_pass":"Pass","baseline_a_fields":"3/3","baseline_a_pass":"Fail"},
 {"script":14,"chatbot_fields":"-","chatbot_pass":"Pass","baseline_a_fields":"-", "baseline_a_pass":"Pass"},
 {"script":15,"chatbot_fields":"-","chatbot_pass":"Pass","baseline_a_fields":"-", "baseline_a_pass":"Pass"},
 {"script":16,"chatbot_fields":"-","chatbot_pass":"Pass","baseline_a_fields":"-", "baseline_a_pass":"Pass"},
 {"script":17,"chatbot_fields":"-","chatbot_pass":"Pass","baseline_a_fields":"-", "baseline_a_pass":"Pass"},
 {"script":18,"chatbot_fields":"-","chatbot_pass":"Pass","baseline_a_fields":"1/1","baseline_a_pass":"Fail"},
 {"script":19,"chatbot_fields":"-","chatbot_pass":"Pass","baseline_a_fields":"-", "baseline_a_pass":"Fail"},
 {"script":20,"chatbot_fields":"-","chatbot_pass":"Pass","baseline_a_fields":"-", "baseline_a_pass":"Fail"},
]
df_cmp = pd.DataFrame(comparison)
print(f"Chatbot: {sum(1 for r in comparison if r['chatbot_pass']=='Pass')}/20 pass   "
      f"Baseline A: {sum(1 for r in comparison if r['baseline_a_pass']=='Pass')}/20 pass")
df_cmp

### Key findings from the independent check

- **0 invented numbers across all 20 chatbot scripts** (vs. Baseline A inventing a share price in script 7 and a full set of fabricated COVID/2022 stress-test figures in script 19) -- the portfolio agent's grounding check (`check_numbers_grounded`) holds up under spot-checking, not just in its own self-report.
- **Script 2 (Persona A) -- genuine bug:** `principal_protection` was misread as "Yes" (true answer: No). Compliance correctly withheld a recommendation given that false belief, but it means the client's valid follow-up ("show me Strategy 2") was never answered -- no portfolio was ever shown in this script.
- **Script 5 (Persona C) -- genuine bug, more serious:** the extracted profile is correct, but the self-correction loop used all 3 attempts on narrow response-wording complaints (disclosure phrasing, not substance) and then withheld the recommendation entirely instead of falling back to the code's own `template_response()` fallback. A fully valid, non-contradictory client never receives a portfolio at all here -- a direct miss against Part 5's "final output must be the number of units to buy or sell" requirement.
- **Scripts 6 and 9 -- minor:** `income` is misread as "$80,000-$99,999" in both -- the chatbot appears to confuse the stated **investment amount** ($80,000) with an income bracket in these condensed continuation scripts, where the client never actually states an income figure. Doesn't block delivery in either script.

## Error-injection test of the compliance / checking agent

Course minimum is 5 deliberately-inserted errors. The team inserted 15 distinct error types (weights not summing to 100%, an unsupported expected-return or projection figure, a mismatched structured claim, holding an excluded asset, exceeding a weight/volatility/bond-floor/commodity cap, exceeding budget, a removed disclaimer, a removed price date, inserted guarantee language, and an unsuitable recommendation for a principal-protection client) x 3 personas = 45 injected errors, plus 3 clean-draft controls.

In [ ]:
with open("evaluation/results/checker_injection_full.md") as f:
    print(f.read())

**45/45 injected errors caught.** One nuance worth noting transparently: of the 3 clean (error-free) draft controls, Persona A's was incorrectly flagged REVISE by the compliance agent's LLM review step -- a false positive, not a missed error, and the only one of the three.

## Mystery client (Nov 30)

**Not run yet** -- the TAs send the mystery client profile Monday. Per the course rules, this notebook must run **exactly as submitted on Sunday**; no code, prompts, or models may be modified afterward. Once the profile arrives, feed its answers through `robo_graph.run_turn()` the same way the live demo above does, and submit the full conversation transcript via Quercus before presentations begin.